# Using Tools

In [ ]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4o-mini"
openai = OpenAI()

In [ ]:
system_message = "You are a helpful assistant for an Airline called FlightAI. "
system_message += "Give short, courteous answers, no more than 1 sentence. "
system_message += "Always be accurate. If you don't know the answer, say so."

In [ ]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool get_ticket_price called for {destination_city}")
    city = destination_city.lower()
    return ticket_prices.get(city, "Unknown")

In [ ]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [ ]:
available_tickets = {"london": "yes", "paris": "yes", "tokyo": "yes", "berlin": "no"}

def get_availability(destination_city):
    print(f"Tool get_availability called for {destination_city}")
    city = destination_city.lower()
    return available_tickets.get(city, "Unknown")

In [ ]:
# get_availability("Berlin")

In [ ]:
availability_function = {
    "name": "get_availability",
    "description": "Get the availability of a return ticket to the destination city. Call this whenever you need to know the availability, for example when a customer asks 'Are there tickest available to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [ ]:
# And this is included in a list of tools:

tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": availability_function}
]

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, city = handle_tool_call(message)
        messages.append(message)
        for response in responses:
            messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        print(messages)
    
    return response.choices[0].message.content

In [ ]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    responses = []
    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        
        if function_name == "get_ticket_price":
            city = arguments.get('destination_city')
            price = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": json.dumps({"destination_city": city,"price": price}),
                "tool_call_id": tool_call.id
            })
    
        elif function_name == "get_availability":
            city = arguments.get('destination_city')
            availabilty = get_availability(city)
            responses.append({
                "role": "tool",
                "content": json.dumps({"destination_city": city,"availability": availabilty}),
                "tool_call_id": tool_call.id
            })
        
    return responses, city

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()